# Backtest — IBEX35 Direction Models

Walk-forward backtest on the out-of-sample period (dates > `train_end` of each artifact).

**Execution convention:** Signal at close of day *t* → enter at open of *t+1* → exit at open of *t+2* (h=1) or *t+6* (h=5).  
Return = `open[t+entry+h] / open[t+entry] − 1`, net of tiered transaction costs.

**Cost tiers (per side):** top-10 market cap 10 bps · mid-10 15 bps · bottom-10 20 bps.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from config import ARTIFACTS_PATH, load_env

load_env()

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

## 0. Load artifacts and OHLCV

In [ ]:
# Load all discrete artifacts that have a train_end
artifacts = {}
for p in sorted(ARTIFACTS_PATH.glob('*.pkl')):
    a = joblib.load(p)
    if a.get('target_type') == 'discrete' and a.get('train_end'):
        key = p.stem
        artifacts[key] = a
        print(f'  {key:35s} train_end={a["train_end"]}  h={a["horizon"]}')

if not artifacts:
    print('No discrete artifacts with train_end found. Train models first.')

In [ ]:
from db.sqlite.queries_ohlcv import fetch_ohlcv
from db.utils_ohlcv import get_ibex_tickers, get_macro_tickers
from models.trees.features import ml_ready

ibex_tickers  = get_ibex_tickers()
macro_tickers = get_macro_tickers()

df_micro_raw = fetch_ohlcv(ibex_tickers)
df_micro_raw = df_micro_raw[df_micro_raw['volume'] > 0].reset_index(drop=True)
df_macro_raw = fetch_ohlcv(macro_tickers)

print(f'OHLCV rows  : {len(df_micro_raw):,}')
print(f'Date range  : {df_micro_raw.date.min().date()} -> {df_micro_raw.date.max().date()}')
print(f'Tickers     : {df_micro_raw.ticker.nunique()}')

In [ ]:
# OHLCV is loaded above; feature matrices are built on demand per (horizon, ft_type).
# See generate_signals() below — it caches by (horizon, ft_type) to avoid redundant rebuilds.
_feature_cache: dict = {}  # (horizon, ft_type) -> (df_meta, X)

def _get_features(horizon: int, ft_type: str):
    key = (horizon, ft_type)
    if key not in _feature_cache:
        df_macro_arg = df_macro_raw if ft_type == "macro" else None
        df_f, X_f, _, mask_f, _ = ml_ready(horizon, df_micro_raw, df_macro=df_macro_arg, ft_type=ft_type)
        meta = df_f.loc[mask_f, ["ticker", "date", "open"]].copy().reset_index(drop=True)
        X    = X_f.reset_index(drop=True)
        _feature_cache[key] = (meta, X)
        print(f"  built features h={horizon} ft={ft_type}: {len(X):,} rows, {X.shape[1]} features")
    return _feature_cache[key]


## 1. Signal generation

For each artifact, run inference on every row where `date > train_end`.
Neural models use the full sequence window; tree models use the flat feature row.

In [ ]:
from models.predict import _predict_tree, _predict_rnn, NEURAL_MODELS

def generate_signals(artifact: dict, stem: str) -> pd.DataFrame:
    horizon   = artifact["horizon"]
    ft_type   = artifact["ft_type"]
    train_end = pd.Timestamp(artifact["train_end"])
    model_key = artifact["model_key"]

    df_meta, X_full = _get_features(horizon, ft_type)
    mask_oos = df_meta["date"] > train_end

    if mask_oos.sum() == 0:
        print(f"  {stem}: no out-of-sample rows")
        return pd.DataFrame()

    if model_key in NEURAL_MODELS:
        preds, probas, last_dates, last_tickers = _predict_rnn(
            artifact, X_full,
            df_meta["ticker"], df_meta["date"], "discrete"
        )
        sig = pd.DataFrame({"ticker": last_tickers, "date": last_dates,
                             "pred": preds, "proba": probas})
        sig = sig[sig["date"] > train_end].reset_index(drop=True)
    else:
        X_oos    = X_full.loc[mask_oos.values]
        meta_oos = df_meta.loc[mask_oos.values].reset_index(drop=True)
        preds, probas = _predict_tree(artifact, X_oos)
        sig = meta_oos[["ticker", "date"]].copy()
        sig["pred"]  = preds
        sig["proba"] = probas

    sig["model"] = stem
    print(f"  {stem:35s} OOS rows={len(sig):,}  "
          f"dates={sig.date.min().date()} -> {sig.date.max().date()}")
    return sig


signals = {}
for stem, a in artifacts.items():
    signals[stem] = generate_signals(a, stem)


## 2. Transaction cost schedule

Tiered by market cap (proxied by mean daily volume).  
Top 10: 10 bps/side · Mid 10: 15 bps · Bottom 10: 20 bps.

In [ ]:
mean_vol = (
    df_micro_raw.groupby('ticker')['volume'].mean()
    .sort_values(ascending=False)
    .reset_index()
)
mean_vol['rank'] = range(1, len(mean_vol) + 1)

def cost_per_side(ticker: str) -> float:
    r = mean_vol.loc[mean_vol['ticker'] == ticker, 'rank']
    rank = int(r.iloc[0]) if not r.empty else 30
    if rank <= 10:  return 0.0010
    if rank <= 20:  return 0.0015
    return 0.0020

cost_map = {t: cost_per_side(t) for t in ibex_tickers}

print('Cost schedule (bps/side):')
for t, c in sorted(cost_map.items(), key=lambda x: x[1]):
    print(f'  {t:12s}  {c*10000:.0f} bps')

## 3. Backtest engine

For each signal day *t*:
- Apply confidence filter `|proba − 0.5| >= (tau − 0.5)` (tau=0.50 = no filter)
- Long stocks with `pred=1`; optionally short `pred=0` stocks
- Enter at open of *t+1*, exit at open of *t+1+h*
- Net return per stock: gross − 2×cost (round trip)
- Portfolio return = equal-weight mean across positions

In [ ]:
# Open-price lookup indexed by (ticker, date)
open_price = (
    df_micro_raw[['ticker', 'date', 'open']]
    .dropna()
    .set_index(['ticker', 'date'])['open']
    .to_dict()
)

all_dates   = sorted(df_micro_raw['date'].unique())
date_to_idx = {d: i for i, d in enumerate(all_dates)}

def next_open(ticker: str, signal_date, steps: int):
    idx = date_to_idx.get(signal_date)
    if idx is None or idx + steps >= len(all_dates):
        return np.nan
    return open_price.get((ticker, all_dates[idx + steps]), np.nan)


def run_backtest(
    sig: pd.DataFrame,
    horizon: int = 1,
    tau: float = 0.50,
    long_short: bool = False,
    min_positions: int = 1,
) -> pd.DataFrame:
    """
    Returns DataFrame with columns: date, port_ret (net equal-weight return).
    """
    records = []
    for date, grp in sig.groupby('date'):
        filtered = grp[np.abs(grp['proba'] - 0.5) >= (tau - 0.5)]
        longs  = filtered[filtered['pred'] == 1]['ticker'].tolist()
        shorts = filtered[filtered['pred'] == 0]['ticker'].tolist() if long_short else []

        if len(longs) < min_positions:
            continue

        day_rets = []
        for tkr in longs:
            p_in  = next_open(tkr, date, 1)
            p_out = next_open(tkr, date, 1 + horizon)
            if not (np.isnan(p_in) or np.isnan(p_out) or p_in == 0):
                day_rets.append(p_out / p_in - 1 - 2 * cost_map.get(tkr, 0.0020))

        for tkr in shorts:
            p_in  = next_open(tkr, date, 1)
            p_out = next_open(tkr, date, 1 + horizon)
            if not (np.isnan(p_in) or np.isnan(p_out) or p_in == 0):
                day_rets.append(-(p_out / p_in - 1) - 2 * cost_map.get(tkr, 0.0020))

        if day_rets:
            records.append({'date': date, 'port_ret': np.mean(day_rets)})

    return pd.DataFrame(records)

## 4. Performance metrics

In [ ]:
RISK_FREE_DAILY = 0.035 / 252   # ~3.5% annual EUR risk-free

def performance_metrics(ret_series: pd.Series, label: str = '') -> dict:
    """Annualised Sharpe (Lo 2002 autocorr-adjusted), max drawdown, Calmar, CAGR."""
    r = ret_series.dropna()
    if len(r) < 10:
        return {}

    excess  = r - RISK_FREE_DAILY
    mean_ex = excess.mean()
    std_ex  = excess.std()

    # Lo (2002): deflate by sqrt of autocorrelation-adjusted variance scaling
    n_lags  = min(5, len(r) // 4)
    rho     = [r.autocorr(lag=k) for k in range(1, n_lags + 1)]
    rho_sum = sum((1 - k / (n_lags + 1)) * rho[k - 1] for k in range(1, n_lags + 1))
    adj     = np.sqrt(max(1 + 2 * rho_sum, 0.01))
    sharpe_raw = (mean_ex / std_ex * np.sqrt(252)) if std_ex > 0 else 0.0
    sharpe_adj = sharpe_raw / adj

    cum   = (1 + r).cumprod()
    dd    = (cum - cum.cummax()) / cum.cummax()
    max_dd = float(dd.min())

    years = len(r) / 252
    cagr  = float((1 + r).prod()) ** (1 / max(years, 0.01)) - 1
    calmar = cagr / abs(max_dd) if max_dd != 0 else np.nan

    return {
        'label':      label,
        'n_days':     len(r),
        'CAGR':       round(cagr, 4),
        'Sharpe':     round(sharpe_raw, 3),
        'Sharpe_adj': round(sharpe_adj, 3),
        'MaxDD':      round(max_dd, 4),
        'Calmar':     round(calmar, 3) if not np.isnan(calmar) else np.nan,
        'hit_rate':   round(float((r > 0).mean()), 4),
    }

## 5. Run backtests — all models, long-only, tau=0.50

In [ ]:
results = {}   # stem -> returns DataFrame (index=date)
metrics = []   # list of metric dicts

for stem, sig in signals.items():
    if sig.empty:
        continue
    h  = artifacts[stem]['horizon']
    bt = run_backtest(sig, horizon=h, tau=0.50, long_short=False)
    if bt.empty:
        print(f'  {stem}: no tradeable days')
        continue
    bt = bt.set_index('date').sort_index()
    results[stem] = bt
    m = performance_metrics(bt['port_ret'], label=stem)
    metrics.append(m)
    print(f'  {stem:35s}  CAGR={m["CAGR"]:+.2%}  '
          f'Sharpe={m["Sharpe"]:+.3f}  Sharpe_adj={m["Sharpe_adj"]:+.3f}  MaxDD={m["MaxDD"]:.2%}')

df_metrics = pd.DataFrame(metrics).sort_values('Sharpe_adj', ascending=False)
display(df_metrics)

## 6. Benchmarks

**B1 — Buy-and-hold IBEX35** · **B2 — Naive momentum** · **B3 — Random long (1 000 bootstraps)**

In [ ]:
# B1: buy-and-hold IBEX35
ibex_raw = (
    df_macro_raw[df_macro_raw['ticker'] == '^IBEX']
    .copy().sort_values('date')
)
ibex_raw['bh_ret'] = ibex_raw['close'].pct_change()

earliest_end = min(pd.Timestamp(a['train_end']) for a in artifacts.values())
ibex_oos = ibex_raw[ibex_raw['date'] > earliest_end].set_index('date').dropna(subset=['bh_ret'])

bh_metrics = performance_metrics(ibex_oos['bh_ret'], label='BuyHold_IBEX35')
print('Buy & Hold:', bh_metrics)

In [ ]:
# B2: naive momentum (long if prior-day return > 0)
h1_stems = [s for s in signals if artifacts[s]['horizon'] == 1]
mom_metrics = {}

if h1_stems:
    ref_dates = signals[h1_stems[0]][['ticker', 'date']]
    mom_df = (
        df_micro_raw.sort_values(['ticker', 'date'])
        .assign(past_ret=lambda d: d.groupby('ticker')['close'].pct_change(1))
        .assign(pred=lambda d: (d['past_ret'] > 0).astype(int))
        .assign(proba=lambda d: (d['past_ret'] > 0).astype(float) * 0.6)
    )
    mom_sig = mom_df.merge(ref_dates, on=['ticker', 'date'])[['ticker', 'date', 'pred', 'proba']]
    bt_mom  = run_backtest(mom_sig, horizon=1, tau=0.50)
    if not bt_mom.empty:
        bt_mom = bt_mom.set_index('date').sort_index()
        results['Momentum_h1'] = bt_mom
        mom_metrics = performance_metrics(bt_mom['port_ret'], label='Momentum_h1')
        print('Naive momentum:', mom_metrics)

In [ ]:
# B3: random long (1 000 bootstrap samples) — null distribution for Sharpe
rng = np.random.default_rng(42)

if h1_stems:
    ref = signals[h1_stems[0]].copy()
    rand_sharpes = []
    for _ in range(1000):
        r_sig = ref.copy()
        r_sig['pred']  = rng.integers(0, 2, size=len(r_sig))
        r_sig['proba'] = 0.5
        bt_r = run_backtest(r_sig, horizon=1, tau=0.50)
        if not bt_r.empty:
            m = performance_metrics(bt_r.set_index('date')['port_ret'])
            if m:
                rand_sharpes.append(m['Sharpe'])

    p5, p50, p95 = np.percentile(rand_sharpes, [5, 50, 95])
    print(f'Random Sharpe distribution: p5={p5:.3f}  median={p50:.3f}  p95={p95:.3f}')
    print(f'Strategy must beat {p95:.3f} to be non-random at 95% confidence')

## 7. Equity curves

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

for stem, bt in results.items():
    cum = (1 + bt['port_ret']).cumprod()
    lw  = 2.0 if stem in ('Momentum_h1',) else 1.2
    ls  = '--' if stem in ('Momentum_h1',) else '-'
    ax.plot(cum.index, cum.values, lw=lw, ls=ls, label=stem)

if not ibex_oos.empty:
    cum_bh = (1 + ibex_oos['bh_ret']).cumprod()
    ax.plot(cum_bh.index, cum_bh.values, lw=2, ls=':', color='black', label='BuyHold_IBEX35')

ax.axhline(1, color='grey', lw=0.7, ls='--')
ax.set_title('Cumulative equity curves — OOS, long-only, tau=0.50, net of costs')
ax.set_ylabel('Growth of 1')
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('{x:.2f}'))
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## 8. Drawdown

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))

for stem, bt in results.items():
    if stem in ('Momentum_h1',):
        continue
    cum  = (1 + bt['port_ret']).cumprod()
    dd   = (cum - cum.cummax()) / cum.cummax()
    ax.plot(dd.index, dd.values * 100, lw=1.0, label=stem)

ax.axhline(0, color='grey', lw=0.7)
ax.set_title('Drawdown (%)')
ax.set_ylabel('DD %')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## 9. Regime analysis

Segment returns by VIX level: calm (<15) · moderate (15–25) · elevated (25–35) · crisis (>35).

> **On CV logs and crisis claims:** A CV window that overlapped a volatile period (e.g. COVID)
> does *not* prove crisis robustness. By window 60/70, COVID data is inside many training sets.
> This section asks a better question: is OOS Sharpe positive across *all* regimes,
> or concentrated in one? Concentrated = fragile.

In [ ]:
vix_raw = (
    df_macro_raw[df_macro_raw['ticker'] == '^VIX'][['date', 'close']]
    .rename(columns={'close': 'vix'})
    .sort_values('date')
)

def vix_regime(v):
    if v < 15:  return 'calm'
    if v < 25:  return 'moderate'
    if v < 35:  return 'elevated'
    return 'crisis'

vix_raw['regime'] = vix_raw['vix'].apply(vix_regime)

regime_rows = []
for stem, bt in results.items():
    if stem in ('Momentum_h1',):
        continue
    merged = bt.reset_index().merge(vix_raw, on='date', how='left')
    for regime, grp in merged.groupby('regime'):
        m = performance_metrics(grp['port_ret'].reset_index(drop=True))
        if m:
            regime_rows.append({'model': stem, 'regime': regime,
                                 'n_days': m['n_days'],
                                 'Sharpe_adj': m['Sharpe_adj'],
                                 'CAGR': m['CAGR'],
                                 'hit_rate': m['hit_rate']})

df_regime = pd.DataFrame(regime_rows).sort_values(['model', 'regime'])
display(df_regime)

In [ ]:
# Heatmap: Sharpe_adj per model x regime
pivot = df_regime.pivot_table(index='model', columns='regime', values='Sharpe_adj')
order = [c for c in ['calm', 'moderate', 'elevated', 'crisis'] if c in pivot.columns]
pivot = pivot[order]

fig, ax = plt.subplots(figsize=(7, max(3, len(pivot) * 0.55 + 1)))
im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn', vmin=-1.5, vmax=1.5)
plt.colorbar(im, ax=ax, label='Sharpe (adj)')
ax.set_xticks(range(len(pivot.columns)));  ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)));    ax.set_yticklabels(pivot.index, fontsize=8)
for i in range(len(pivot.index)):
    for j, col in enumerate(pivot.columns):
        v = pivot.loc[pivot.index[i], col]
        if not np.isnan(v):
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=8,
                    color='white' if abs(v) > 1 else 'black')
ax.set_title('Sharpe (autocorr-adj) by VIX regime')
plt.tight_layout()
plt.show()

## 10. Confidence filter sweep

Sweep tau for the best model (by Sharpe_adj at tau=0.50).  
Tau must be chosen from CV evidence, **not** by maximising backtest Sharpe.

In [ ]:
if not df_metrics.empty:
    best_stem = df_metrics.iloc[0]['label']
    best_sig  = signals[best_stem]
    best_h    = artifacts[best_stem]['horizon']

    tau_rows = []
    for tau in [0.50, 0.52, 0.55, 0.58, 0.60]:
        bt = run_backtest(best_sig, horizon=best_h, tau=tau, long_short=False, min_positions=5)
        if bt.empty:
            continue
        n_active = bt['date'].nunique() if 'date' in bt.columns else len(bt)
        coverage = len(bt) / max(best_sig['date'].nunique(), 1)
        m = performance_metrics(bt.set_index('date')['port_ret'] if 'date' in bt.columns
                                 else bt['port_ret'])
        if m:
            tau_rows.append({'tau': tau, 'coverage': round(coverage, 3),
                             'Sharpe_adj': m['Sharpe_adj'], 'CAGR': m['CAGR'],
                             'MaxDD': m['MaxDD'], 'hit_rate': m['hit_rate']})

    df_tau = pd.DataFrame(tau_rows)
    print(f'Model: {best_stem}')
    display(df_tau)

## 11. Long-only vs Long/Short (best model)

In [ ]:
if not df_metrics.empty:
    variants = [
        ('Long-only  tau=0.50', False, 0.50),
        ('Long/Short tau=0.50', True,  0.50),
        ('Long/Short tau=0.55', True,  0.55),
    ]
    fig, ax = plt.subplots(figsize=(11, 4))
    for label, ls_flag, tau in variants:
        bt = run_backtest(best_sig, horizon=best_h, tau=tau, long_short=ls_flag)
        if bt.empty:
            continue
        cum = (1 + bt.set_index('date').sort_index()['port_ret']).cumprod()
        ax.plot(cum.index, cum.values, lw=1.5, label=label)

    ax.axhline(1, color='grey', lw=0.7, ls='--')
    ax.set_title(f'Long-only vs Long/Short — {best_stem}')
    ax.set_ylabel('Growth of 1')
    ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('{x:.2f}'))
    ax.legend()
    plt.tight_layout()
    plt.show()

## 12. Summary

In [ ]:
all_m = list(metrics)
if bh_metrics:  all_m.append(bh_metrics)
if mom_metrics: all_m.append(mom_metrics)

df_summary = (
    pd.DataFrame(all_m)
    .set_index('label')
    [['CAGR', 'Sharpe', 'Sharpe_adj', 'MaxDD', 'Calmar', 'hit_rate', 'n_days']]
    .sort_values('Sharpe_adj', ascending=False)
)
df_summary.style \
    .format({'CAGR': '{:+.2%}', 'Sharpe': '{:.3f}', 'Sharpe_adj': '{:.3f}',
             'MaxDD': '{:.2%}', 'Calmar': '{:.3f}', 'hit_rate': '{:.3f}'}) \
    .background_gradient(subset=['Sharpe_adj'], cmap='RdYlGn', vmin=-1, vmax=1)